# Experiment 2 — Geometry: Phase 2 (model runs, ReAct, v2)

Runs every pipeline from Phase 1 under the four state-passing conditions and
the two propagation modes, one model per notebook run.

**Input** `geometry_exp2_pipelines.json` (Phase 1)
**Output** `results/<model>/results.jsonl` (append-only, resumable),
`results.json` (latest record per unit), `manifest.json`, `summary_report.txt`,
`errors.md`, `llm_io.jsonl`

## Protocol

* **ReAct**: one `Thought:` line, one `Action:` JSON line. Temperature 0.
  **Hidden reasoning off** (direct answer, as in Exp1 and the protocol); the
  chain-of-thought ablation is a separate run with `THINKING = "on"`. A step
  whose reply is empty at `max_tokens` is recorded as `budget_exhausted`,
  separately from parse failures.
* **No property read tools** in the four primary conditions. The model sees
  exactly what Table 3 says it sees. Under `handle` and `handle_sum` it may
  call `get_object(id)`, which returns the WKT of that handle and nothing
  else (no property), does not consume the step, and is budgeted at one call
  per visible object. A property can therefore never be *read*; it can only
  be reconstructed from text the model asked for. Fetch rate is recorded.
* **Gated steps name the admissible calls.** A call to any other tool is a
  *deferral* (recorded, scored as incorrect, reported separately from a wrong
  decision). Under cascading mode the deferred call is still executed, because
  that is what would happen in a real agent loop.
* **Oracle mode**: the correct call is applied after every step regardless of
  what the model said, so per-step accuracy isolates reconstruction burden.
* **Cascading mode**: the model's call is executed; a malformed call aborts
  the pipeline.
* **Experiment 4** (`EXP4_DESCRIBE = True`): adds a `describe(polygon)` read
  tool under `raw` and `handle`, budget `n_visible + 1` per step. Records
  carry `protocol = "react_describe_v1"` and describe-call logs. Never mix
  Exp4 results with the primary run.

The record schema is a superset of v1, so the existing Phase 3 evaluation
notebook loads these files unchanged; the new fields are `deferral`,
`decision_correct`, `margin_band`, `allowed_tools`, `get_object_calls`.

## [CELL 1] Configuration — model registry

Change **`USE_MODEL`** only. Everything else (provider, endpoint, key name,
thinking switch, max_tokens, results folder) is derived from the preset.

To add a model, copy one block inside `PRESETS` and edit the fields. Each model
writes to its own `results/<USE_MODEL>/` folder, and the CoT ablation
(`THINKING = "on"`) writes to `results/<USE_MODEL>__cot/`, so runs never
overwrite each other.


In [ ]:
!pip install -q shapely openai tenacity

import os, pathlib, re

# ===== 1. PICK THE MODEL ==================================================
USE_MODEL = "gpt-4.1-mini"        # <-- change ONLY this line to swap models
# USE_MODEL = "mock"  
THINKING  = "off"                      # "off" = primary run | "on" = CoT ablation (Table 5)

# ===== 2. MODEL REGISTRY ==================================================
# To add a model: copy a block, change the fields. Nothing else in the notebook changes.
#   provider   : "openai_compatible" | "google" | "mock_oracle" | "mock_random"
#   base_url   : OpenAI-compatible endpoint (ignored by the google provider)
#   key_env    : env var / .env key holding the API key
#   off / on   : how THIS provider switches hidden reasoning off / on
#                openai_compatible -> extra_body dict ; google -> thinking_budget int
PRESETS = {
    "deepseek-v4-flash": dict(
        model="deepseek-v4-flash", provider="openai_compatible",
        base_url="https://api.deepseek.com", key_env="DEEPSEEK_API_KEY",
        off={"thinking": {"type": "disabled"}}, on={}, max_tokens=4096),

    "deepseek-chat": dict(
        model="deepseek-chat", provider="openai_compatible",
        base_url="https://api.deepseek.com", key_env="DEEPSEEK_API_KEY",
        off={"thinking": {"type": "disabled"}}, on={}, max_tokens=4096),

    "gemini-2.5-flash": dict(
        model="gemini-2.5-flash", provider="google",
        base_url=None, key_env="GEMINI_API_KEY",
        off=0, on=-1, max_tokens=4096),          # google: thinking_budget, 0=off, -1=dynamic

    "gemini-2.5-flash-lite": dict(
        model="gemini-2.5-flash-lite", provider="google",
        base_url=None, key_env="GEMINI_API_KEY",
        off=0, on=-1, max_tokens=4096),

    "gpt-4.1-mini": dict(
        model="gpt-4.1-mini", provider="openai_compatible",
        base_url=None, key_env="OPENAI_API_KEY",
        off={}, on={}, max_tokens=4096),

    "qwen3-openrouter": dict(
        model="qwen/qwen3-32b", provider="openai_compatible",
        base_url="https://openrouter.ai/api/v1", key_env="OPENROUTER_API_KEY",
        # qwen3-32b is served by BOTH DeepInfra and SiliconFlow. Only SiliconFlow honours
        # reasoning.enabled=False; DeepInfra ignores it and still spends ~300 reasoning
        # tokens per step. OpenRouter routes dynamically, so without pinning you get a
        # SILENT mix of thinking-on and thinking-off steps in the same run.
        # allow_fallbacks=False: fail loudly rather than fall back to DeepInfra.
        off={"reasoning": {"enabled": False},
             "provider": {"order": ["SiliconFlow"], "allow_fallbacks": False}},
        on={}, max_tokens=4096),

    "mock": dict(                                 # free dry-run, no API key needed
        model="mock_oracle", provider="mock_oracle",
        base_url=None, key_env=None, off={}, on={}, max_tokens=1024),
}

# ===== 3. APPLY THE PRESET (do not edit below) ============================
if USE_MODEL not in PRESETS:
    raise KeyError(f"unknown model {USE_MODEL!r}. available: {sorted(PRESETS)}")
if THINKING not in ("off", "on"):
    raise ValueError(f"THINKING must be 'off' or 'on', got {THINKING!r}")

_P = PRESETS[USE_MODEL]
MODEL        = _P["model"]
PROVIDER     = _P["provider"]
BASE_URL     = _P["base_url"]
API_KEY_ENV  = _P["key_env"]
MAX_TOKENS   = _P["max_tokens"]
_switch      = _P[THINKING]
EXTRA_BODY      = _switch if PROVIDER == "openai_compatible" else {}
THINKING_BUDGET = _switch if PROVIDER == "google" else 0
# each model gets its own results folder; the CoT ablation never overwrites the primary run
RUN_TAG = re.sub(r"[^A-Za-z0-9_.-]", "_", USE_MODEL) + ("" if THINKING == "off" else "__cot")

# ===== 4. RUN SETTINGS (same for every model) =============================
DATASET      = "geometry_exp2_pipelines.json"
RUN_CONDITIONS = ["raw", "augmented", "handle", "handle_sum"]
RUN_MODES      = ["cascading", "oracle"]
EXP4_DESCRIBE  = False                 # True -> Exp 4; use RUN_CONDITIONS = ["raw", "handle"]
GET_OBJECT_IN_HANDLE_CONDITIONS = True
WORKERS      = 4
SMOKE        = 0                       # >0 -> run only this many pipelines (stratified smoke test)
TEMPERATURE  = 0.0
FORCE        = False                   # True -> ignore existing results and re-run everything
LOG_LLM_IO   = True
STOP_ON_RATE_LIMIT = True

# ===== 5. API KEY: env var, else .env next to the notebook ================
def _read_dotenv(name, override=True):
    """Find `name` in a .env file: cwd, two parents up, or beside this notebook."""
    seen = []
    here = pathlib.Path.cwd()
    for d in [here] + list(here.parents)[:2]:      # list(): .parents slicing needs py3.10
        p = d / ".env"
        if p.is_file() and p not in seen:
            seen.append(p)
    for p in seen:
        for raw in p.read_text(encoding="utf-8").splitlines():
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[len("export "):].strip()
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip()
            if len(v) >= 2 and v[0] == v[-1] and v[0] in "\"'":
                v = v[1:-1]
            if k == name and v:
                if override or not os.environ.get(k):
                    os.environ[k] = v
                return v
    return None

def _get_key():
    if PROVIDER.startswith("mock"):
        return "mock"
    k = os.environ.get(API_KEY_ENV) or _read_dotenv(API_KEY_ENV)
    if not k:
        try:
            from google.colab import userdata
            k = userdata.get(API_KEY_ENV)
        except Exception:
            k = None
    if not k:
        raise RuntimeError(
            f"{API_KEY_ENV} not found for model {USE_MODEL!r}.\n"
            f"Put '{API_KEY_ENV}=...' in a .env file next to this notebook "
            f"(cwd is {pathlib.Path.cwd()}), or export it before starting Jupyter."
        )
    return k

# ===== 6. SHOW WHAT WILL RUN =============================================
_k = _get_key()
print(f"model      : {USE_MODEL}  ->  {MODEL}")
print(f"provider   : {PROVIDER}   base_url={BASE_URL}")
print(f"key        : " + ("not needed (mock)" if PROVIDER.startswith("mock")
                        else f"{API_KEY_ENV} present, ends ...{_k[-4:]}"))
print(f"thinking   : {THINKING}   extra_body={EXTRA_BODY}  thinking_budget={THINKING_BUDGET}")
print(f"max_tokens : {MAX_TOKENS}")
print(f"results    : results/{RUN_TAG}/")
del _k


model      : gpt-4.1-mini  ->  gpt-4.1-mini
provider   : openai_compatible   base_url=None
key        : OPENAI_API_KEY present, ends ...ifAA
thinking   : off   extra_body={}  thinking_budget=0
max_tokens : 4096
results    : results/gpt-4.1-mini/


## [CELL 2] Runtime (identical to the Phase 1 cell)

In [4]:
%%writefile exp2_runtime.py
"""
Experiment 2 - Geometry runtime (v2).  Written identically by the Phase 1 and
Phase 2 notebooks so pipeline construction and pipeline execution share one
definition of every operation, condition and scoring rule.

Changes from v1 (see PHASE1_README):
  * No auxiliary read tools in the four primary conditions.  Table 3 defines
    handle-only as "object id; no property information"; raw and augmented
    must force the model to reconstruct properties from text.  A `describe`
    read tool exists only for Experiment 4 and is switched on explicitly.
  * Gated steps name the admissible tools.  A call to any other tool is a
    DEFERRAL (the model tried to make the runtime compute the property) and is
    scored separately from a wrong decision.
  * buffer uses mitre joins so the vertex count, and therefore the tier, is
    preserved along a pipeline.
"""
import math
import random
import re

from shapely import affinity, delaunay_triangles, voronoi_polygons
from shapely import wkt as shapely_wkt
from shapely.geometry import MultiPoint, Polygon
from shapely.geometry.polygon import orient
from shapely.ops import nearest_points as _nearest_points

WKT_PRECISION = 0            # integer grid: every object, initial or intermediate, is serialized with integer coordinates

# ===========================================================================
# Polygon generator (identical to Experiment 1 Phase 1 v3)
# ===========================================================================


# A tier is (vmin, vmax, coord_lo, coord_hi, coord_type).  coord_type "int"
# rounds to integers, "float2" to two decimals.
TIERS = {"simple": (3, 8, 0.0, 1000.0, "int"),      # protocol vertex ranges, integer grid in every tier
         "medium": (10, 20, 0.0, 1000.0, "int"),
         "hard":   (20, 40, 0.0, 1000.0, "int")}
PRECISION = WKT_PRECISION
# Target area is a FRACTION of the coordinate span squared, log-uniform, so it
# is matched across tiers whether or not the tiers share a coordinate range.
AREA_FRAC_LO, AREA_FRAC_HI = 0.004, 0.12
# Aspect-ratio targets (bbox width / height), log-uniform.  Irregular polygons
# are elongated (stretch 2-5 on one axis, either axis) as in the protocol.
ASPECT_RANGE = {"convex": (0.5, 2.0), "concave": (0.5, 2.0), "irregular": (2.0, 5.0)}
MIN_FILL = 0.05                              # area / bbox area
MIN_HULL_DEFICIT = 0.03                      # non-convex polygons must miss >= 3% of their hull area


# ---------------------------------------------------------------------------
# Unit-scale shape generators.  All take (rng, n, bias) and return a Polygon
# centred near the origin.  `bias` in [0, 0.5] pushes the centroid away from
# the bbox centre in a random direction (limacon r = 1 + bias*cos(theta)).
# ---------------------------------------------------------------------------

def _angles(rng, n, alpha=0.7):
    """n angles around the circle.  Gaps are a mix of a uniform share and a
    random exponential share, so the minimum gap is at least (1-alpha)*2pi/n
    (no near-coincident vertices) while edge lengths still vary a lot."""
    e = [rng.expovariate(1.0) for _ in range(n)]
    tot = sum(e)
    gaps = [2 * math.pi * ((1 - alpha) / n + alpha * ei / tot) for ei in e]
    start = rng.uniform(0, 2 * math.pi)
    ang, t = [], start
    for g in gaps[:-1]:
        t += g
        ang.append(t % (2 * math.pi))
    ang.append(start)
    return sorted(ang)


def _limacon(theta, bias, phi):
    return 1.0 + bias * math.cos(theta - phi)


def unit_convex(rng, n, bias):
    """Vertices on the convex limacon r = 1 + b cos(t), b <= 0.5, in angular
    order: exact vertex count, guaranteed convex."""
    ang = _angles(rng, n)
    if ang is None:
        return None
    phi = rng.uniform(0, 2 * math.pi)
    return Polygon([(_limacon(t, bias, phi) * math.cos(t),
                     _limacon(t, bias, phi) * math.sin(t)) for t in ang])


def unit_concave(rng, n, bias):
    """Star-shaped by angular sweep with random radii; mild dents."""
    ang = _angles(rng, n)
    if ang is None:
        return None
    phi = rng.uniform(0, 2 * math.pi)
    rmin = rng.uniform(0.40, 0.75)
    return Polygon([(_limacon(t, bias, phi) * rng.uniform(rmin, 1.0) * math.cos(t),
                     _limacon(t, bias, phi) * rng.uniform(rmin, 1.0) * math.sin(t)) for t in ang])


def unit_irregular(rng, n, bias):
    """Protocol class 'irregular/elongated': radial generation with non-uniform
    vertex spacing and radii, followed (in place()) by an affine stretch of
    2-5 on one axis.  No spikes: every vertex sits on the same body, so the
    bbox diagonal is set by the body and not by a few outliers."""
    ang = _angles(rng, n, alpha=0.9)          # strongly non-uniform spacing
    phi = rng.uniform(0, 2 * math.pi)
    rmin = rng.uniform(0.45, 0.85)
    pts = []
    for t in ang:
        r = rng.uniform(rmin, 1.0) * _limacon(t, bias, phi)
        pts.append((r * math.cos(t), r * math.sin(t)))
    return Polygon(pts)


def unit_convex_sharp(rng, n, bias):
    """Points spread over the rounded corners of a random triangle or
    quadrilateral: exactly n vertices, convex, and as 'pointed' as `bias`
    makes it (bias 0 = nearly round, bias 0.5 = nearly the base polygon).
    Round shapes with many vertices have their centroid glued to the bbox
    centre; this generator is what lets a 40-vertex convex polygon have a
    centroid well away from it."""
    m = rng.choice([3, 3, 4])
    base = None
    for _ in range(30):
        pts = [(math.cos(t), math.sin(t)) for t in _angles(rng, m, alpha=0.6)]
        b = Polygon(pts)
        if b.is_valid and b.area > 0.4:
            base = b
            break
    if base is None:
        return None
    r = 0.95 - 1.4 * bias                        # corner radius 0.25..0.95 (tighter collapses on an integer grid)
    ring = list(base.exterior.coords)[:-1]
    arcs = []
    for i in range(m):
        p0, p1, p2 = ring[i - 1], ring[i], ring[(i + 1) % m]
        a0 = math.atan2(p1[1] - p0[1], p1[0] - p0[0]) - math.pi / 2   # outward normal of edge in
        a1 = math.atan2(p2[1] - p1[1], p2[0] - p1[0]) - math.pi / 2   # outward normal of edge out
        turn = (a1 - a0) % (2 * math.pi)
        arcs.append((p1, a0, turn))
    total = sum(a[2] for a in arcs)
    pts = []
    # allocate vertices to arcs in proportion to turn angle, at least one each
    alloc = [1] * m
    for _ in range(n - m):
        u = rng.uniform(0, total); acc = 0
        for i, a in enumerate(arcs):
            acc += a[2]
            if u <= acc:
                alloc[i] += 1
                break
        else:
            alloc[-1] += 1
    for (c, a0, turn), k in zip(arcs, alloc):
        # evenly spaced along the arc with jitter, so no two vertices collapse on the grid
        for i in range(k):
            t = (i + 0.5 + rng.uniform(-0.3, 0.3)) / k
            ang = a0 + t * turn
            pts.append((c[0] + r * math.cos(ang), c[1] + r * math.sin(ang)))
    hull = Polygon(pts).convex_hull
    if hull.geom_type != "Polygon" or len(hull.exterior.coords) - 1 != n:
        return None
    return hull


def projective_squeeze(poly, k, phi):
    """(x, y) -> (x, y) / (1 + k x) after rotating by -phi.  Projective maps
    preserve convexity and simplicity; this one turns a round shape into an
    egg and moves the centroid away from the bbox centre by an amount that
    grows with k.  The polygon is first normalised so |coord| <= 1."""
    ring = list(poly.exterior.coords)[:-1]
    R = max(math.hypot(x, y) for x, y in ring)
    c, s = math.cos(phi), math.sin(phi)
    out = []
    for x, y in ring:
        x, y = x / R, y / R
        u, v = c * x + s * y, -s * x + c * y
        d = 1 + k * u
        u, v = u / d, v / d
        out.append((c * u - s * v, s * u + c * v))
    return Polygon(out)


GENERATORS = {"convex": unit_convex, "concave": unit_concave, "irregular": unit_irregular}


# ---------------------------------------------------------------------------
# Placement: stretch to target aspect, rescale to target area, translate into
# bounds, round to PRECISION.  The rounded polygon is what gets validated.
# ---------------------------------------------------------------------------

def _round(v, ctype):
    return float(round(v)) if ctype == "int" else round(v, PRECISION)


def place(poly, target_area, target_aspect, rng, tier="hard"):
    vmin, vmax, lo, hi, ctype = TIERS[tier]
    if poly is None or poly.is_empty or poly.area <= 0:
        return None
    minx, miny, maxx, maxy = poly.bounds
    w, h = maxx - minx, maxy - miny
    if w <= 0 or h <= 0:
        return None
    poly = affinity.scale(poly, xfact=target_aspect / (w / h), yfact=1.0, origin="centroid")
    s = math.sqrt(target_area / poly.area)
    poly = affinity.scale(poly, xfact=s, yfact=s, origin="centroid")
    minx, miny, maxx, maxy = poly.bounds
    if (maxx - minx) >= (hi - lo) or (maxy - miny) >= (hi - lo):
        return None
    poly = affinity.translate(poly, xoff=rng.uniform(lo - minx, hi - maxx),
                                    yoff=rng.uniform(lo - miny, hi - maxy))
    return Polygon([(_round(x, ctype), _round(y, ctype)) for x, y in list(poly.exterior.coords)[:-1]])


# ---------------------------------------------------------------------------
# Validity gate.  Every rejection has a named reason.
# ---------------------------------------------------------------------------

def is_convex(poly, tol=1e-9):
    a = poly.area
    return a > 0 and abs(poly.convex_hull.area - a) <= tol * max(a, 1.0)


def check_validity(poly, tier, expected_convex):
    vmin, vmax, lo, hi, ctype = TIERS[tier]
    if poly is None:
        return False, "generation_failed"
    if not poly.is_valid:
        return False, "not_valid"
    if not poly.is_simple:
        return False, "not_simple"
    ring = list(poly.exterior.coords)[:-1]
    n = len(ring)
    if not (vmin <= n <= vmax):
        return False, "vertex_count_out_of_range"
    for i in range(n):
        if ring[i] == ring[(i + 1) % n]:
            return False, "duplicate_adjacent"
    crosses = []
    for i in range(n):
        (x0, y0), (x1, y1), (x2, y2) = ring[i - 1], ring[i], ring[(i + 1) % n]
        crosses.append((x1 - x0) * (y2 - y1) - (y1 - y0) * (x2 - x1))
    if min(abs(c) for c in crosses) < 1e-3:
        return False, "collinear"
    sign_convex = all(c > 0 for c in crosses) or all(c < 0 for c in crosses)
    if poly.area <= 10:
        return False, "area_too_small"
    minx, miny, maxx, maxy = poly.bounds
    if min(minx, miny) < lo or max(maxx, maxy) > hi:
        return False, "out_of_bounds"
    fr = poly.area / ((maxx - minx) * (maxy - miny))
    if fr < MIN_FILL:
        return False, "sliver_low_fill"
    if fr >= FILL_MAX:
        return False, "too_round_high_fill"
    if is_convex(poly) != expected_convex or sign_convex != expected_convex:
        return False, "convexity_mismatch"
    if not expected_convex and 1 - poly.area / poly.convex_hull.area < MIN_HULL_DEFICIT:
        return False, "dent_too_shallow"
    return True, None


# ---------------------------------------------------------------------------
# Nuisance quantities used for cross-tier matching.
# ---------------------------------------------------------------------------

def bbox_diagonal(poly):
    minx, miny, maxx, maxy = poly.bounds
    return math.hypot(maxx - minx, maxy - miny)


def centroid_offset_norm(poly):
    """Distance from the true centroid to the bbox centre, / bbox diagonal.
    This is the quantity that decides whether 'answer the middle of the box'
    passes a centroid question at a given tolerance."""
    minx, miny, maxx, maxy = poly.bounds
    c = poly.centroid
    return math.hypot(c.x - (minx + maxx) / 2, c.y - (miny + maxy) / 2) / bbox_diagonal(poly)


def fill_ratio(poly):
    minx, miny, maxx, maxy = poly.bounds
    return poly.area / ((maxx - minx) * (maxy - miny))


# Guessability bands.  Each (tier, shape) cell must contain the same number of
# polygons in each band, so no tier is more guessable than another.
#   offset band: distance centroid -> bbox centre, / bbox diagonal (centroid guessability)
#   fill band:   area / bbox area (area and perimeter guessability from the bbox)
OFFSET_BANDS = [(0.00, 0.02), (0.02, 0.04), (0.04, 0.07), (0.07, 10.0)]
FILL_BANDS = [(0.0, 0.52), (0.52, 0.62), (0.62, 0.70), (0.70, 0.76)]
FILL_MAX = 0.76                              # near-ellipses (fill -> pi/4) are excluded in every tier


def fill_band(fr):
    for i, (lo, hi) in enumerate(FILL_BANDS):
        if lo <= fr < hi:
            return i
    return len(FILL_BANDS) - 1


# Joint (offset band, fill band) cells and their share of each shape cell.
# Only cells reachable in EVERY tier are used (measured on the integer grid:
# e.g. a 40-vertex convex polygon cannot have both a far-off centroid and a
# low fill).  The same shares apply in all three tiers, so no tier is more
# guessable than another on either quantity.
JOINT_SHARES = {
    # convex: a 20-40 vertex convex polygon on the integer grid cannot have fill < 0.52,
    # so convex cells never use fill band 0 in any tier (triangles appear in the irregular class).
    "convex":    {(0, 1): 5, (0, 2): 5, (0, 3): 4, (1, 1): 6, (1, 2): 8, (1, 3): 4, (2, 1): 6, (2, 2): 7, (2, 3): 3, (3, 2): 2},
    "concave":   {(0, 1): 2, (0, 2): 2, (1, 0): 2, (1, 1): 4, (1, 2): 3, (1, 3): 1, (2, 0): 2, (2, 1): 4, (2, 2): 2, (3, 0): 3},
    "irregular": {(0, 1): 2, (0, 2): 2, (1, 0): 2, (1, 1): 4, (1, 2): 3, (1, 3): 1, (2, 0): 2, (2, 1): 4, (2, 2): 2, (3, 0): 3},
}


def joint_quota(shape, n):
    """Scale JOINT_SHARES[shape] to n slots (largest-remainder rounding)."""
    sh = JOINT_SHARES[shape]; tot = sum(sh.values())
    raw = {k: v * n / tot for k, v in sh.items()}
    q = {k: int(v) for k, v in raw.items()}
    for k, _ in sorted(raw.items(), key=lambda kv: -(kv[1] - int(kv[1])))[: n - sum(q.values())]:
        q[k] += 1
    return q


def joint_band(poly):
    return (offset_band(centroid_offset_norm(poly)), fill_band(fill_ratio(poly)))


def offset_band(d):
    for i, (lo, hi) in enumerate(OFFSET_BANDS):
        if lo <= d < hi:
            return i
    return len(OFFSET_BANDS) - 1


def span(tier):
    return TIERS[tier][3] - TIERS[tier][2]


def draw_targets(rng, shape):
    """(area fraction of span^2, aspect ratio) from the shared distributions."""
    af = math.exp(rng.uniform(math.log(AREA_FRAC_LO), math.log(AREA_FRAC_HI)))
    lo, hi = ASPECT_RANGE[shape]
    ar = math.exp(rng.uniform(math.log(lo), math.log(hi)))
    if shape == "irregular" and rng.random() < 0.5:
        ar = 1.0 / ar                      # stretch along either axis
    return af, ar


def make_one(rng, shape, tier, target_area, target_aspect, tries=300):
    """Rejection-sample one valid polygon of the given shape class and tier.
    The two shape knobs (limacon bias, projective squeeze) are drawn at random
    on every attempt from ranges shared across tiers.  Returns (poly, knobs,
    rejection_reasons)."""
    vmin, vmax = TIERS[tier][:2]
    reasons = {}
    for _ in range(tries):
        n = rng.randint(vmin, vmax)
        bias = rng.uniform(0.0, 0.5)
        squeeze = rng.uniform(0.0, 0.85)
        if shape == "convex" and rng.random() < 0.6:
            unit = unit_convex_sharp(rng, n, bias); gen = "convex_sharp"
        else:
            unit = GENERATORS[shape](rng, n, bias); gen = shape
        if unit is None:
            reasons["generation_failed"] = reasons.get("generation_failed", 0) + 1
            continue
        unit = projective_squeeze(unit, squeeze, rng.uniform(0, 2 * math.pi))
        p = place(unit, target_area, target_aspect, rng, tier)
        ok, why = check_validity(p, tier, shape == "convex")
        if ok:
            p = orient(p, 1.0)   # canonical ccw; Phase 1 flips exactly half
            return p, {"generator": gen, "bias": round(bias, 4), "squeeze": round(squeeze, 4)}, reasons
        reasons[why] = reasons.get(why, 0) + 1
    return None, None, reasons


# ===========================================================================
# The twelve specification operations
# ===========================================================================

def op_convex_hull(p):        return p.convex_hull
def op_buffer(p, distance):   return p.buffer(distance, join_style="mitre", mitre_limit=5.0)
def op_simplify(p, tol):      return p.simplify(tol, preserve_topology=True)
def op_rotate(p, angle):      return affinity.rotate(p, angle, origin="centroid")
def op_translate(p, dx, dy):  return affinity.translate(p, xoff=dx, yoff=dy)
def op_scale(p, factor):      return affinity.scale(p, xfact=factor, yfact=factor, origin="centroid")
def op_intersection(a, b):    return a.intersection(b)
def op_union(a, b):           return a.union(b)
def op_centroid(p):           return p.centroid
def op_triangulate(p):        return delaunay_triangles(MultiPoint(_verts(p)))
def op_voronoi(p):            return voronoi_polygons(MultiPoint(_verts(p)))
def op_nearest_points(a, b):  return _nearest_points(a, b)

def op_bounding_box(p):
    a, b, c, d = p.bounds
    return Polygon([(a, b), (c, b), (c, d), (a, d)])

def _verts(p):
    return list(p.exterior.coords)[:-1]


CHAIN_UNARY, CHAIN_BINARY, TERMINAL = "chain_unary", "chain_binary", "terminal"

OPS = {
    "convex_hull":    (op_convex_hull,    CHAIN_UNARY,  1, []),
    "buffer":         (op_buffer,         CHAIN_UNARY,  1, ["distance"]),
    "simplify":       (op_simplify,       CHAIN_UNARY,  1, ["tolerance"]),
    "bounding_box":   (op_bounding_box,   CHAIN_UNARY,  1, []),
    "rotate":         (op_rotate,         CHAIN_UNARY,  1, ["angle"]),
    "translate":      (op_translate,      CHAIN_UNARY,  1, ["dx", "dy"]),
    "scale":          (op_scale,          CHAIN_UNARY,  1, ["factor"]),
    "intersection":   (op_intersection,   CHAIN_BINARY, 2, []),
    "union":          (op_union,          CHAIN_BINARY, 2, []),
    "centroid":       (op_centroid,       TERMINAL,     1, []),
    "triangulate":    (op_triangulate,    TERMINAL,     1, []),
    "voronoi":        (op_voronoi,        TERMINAL,     1, []),
    "nearest_points": (op_nearest_points, TERMINAL,     2, []),
}
SPEC_OPS      = list(OPS)
CHAINABLE_OPS = [k for k, v in OPS.items() if v[1] in (CHAIN_UNARY, CHAIN_BINARY)]
TERMINAL_OPS  = [k for k, v in OPS.items() if v[1] == TERMINAL]
DESCRIBE_TOOL = "describe"          # Experiment 4 only

def n_operands(t):  return OPS[t][2]
def arg_names(t):   return OPS[t][3]
def is_terminal(t): return OPS[t][1] == TERMINAL


def to_wkt(poly, precision=WKT_PRECISION):
    if precision == 0:
        body = ", ".join(f"{int(round(x))} {int(round(y))}" for x, y in poly.exterior.coords)
    else:
        body = ", ".join(f"{x:.{precision}f} {y:.{precision}f}" for x, y in poly.exterior.coords)
    return f"POLYGON(({body}))"


def snap(poly):
    """Round to serialization precision so the stored form is the computed form."""
    return shapely_wkt.loads(to_wkt(poly))


def is_runtime_valid(poly):
    try:
        return (poly is not None and not poly.is_empty and poly.geom_type == "Polygon"
                and poly.is_valid and poly.is_simple and poly.area > 0)
    except Exception:
        return False


def apply_op(tool, operands, args, strict=False):
    """strict=True (pipeline construction): a non-Polygon result is an error
    rather than being replaced by its convex hull."""
    fn, klass, n_ops, names = OPS[tool]
    if len(operands) != n_ops:
        raise ValueError(f"{tool} takes exactly {n_ops} object argument(s), got {len(operands)}")
    if len(args) != len(names):
        raise ValueError(f"{tool} takes exactly {len(names)} numeric argument(s), got {len(args)}")
    out = fn(*operands, *args)
    if klass == TERMINAL:
        return out
    if out.geom_type != "Polygon":
        if strict:
            raise ValueError(f"{tool} produced {out.geom_type}")
        out = out.convex_hull
    return snap(orient(out, 1.0))


# ===========================================================================
# Properties and the deterministic summary (Table 3)
# ===========================================================================

def properties(poly):
    minx, miny, maxx, maxy = poly.bounds
    ring = list(poly.exterior.coords)
    edges = [math.dist(ring[i], ring[i + 1]) for i in range(len(ring) - 1)]
    mean_e = sum(edges) / len(edges)
    w, h = maxx - minx, maxy - miny
    return {
        "vertex_count": len(ring) - 1,
        "area": round(poly.area, 4),
        "perimeter": round(poly.length, 4),
        "width": round(w, 4),
        "height": round(h, 4),
        "aspect_ratio": round(w / h, 4),
        "edge_length_variance": round(sum((e - mean_e) ** 2 for e in edges) / len(edges), 4),
        "bbox": [round(minx, 4), round(miny, 4), round(maxx, 4), round(maxy, 4)],
        "centroid": [round(poly.centroid.x, 4), round(poly.centroid.y, 4)],
        "centroid_y": round(poly.centroid.y, 4),
        "convex": bool(is_convex(poly)),
        "is_simple": bool(poly.is_simple),
    }


GATE_VALUE = {"area": lambda q: q["area"], "perimeter": lambda q: q["perimeter"],
              "aspect_ratio": lambda q: q["aspect_ratio"],
              "edge_length_variance": lambda q: q["edge_length_variance"],
              "centroid_y": lambda q: q["centroid_y"],
              "width": lambda q: q["width"], "vertex_count": lambda q: q["vertex_count"]}
GATE_PHRASE = {"area": "area", "perimeter": "perimeter",
               "aspect_ratio": "aspect ratio (bounding-box width divided by bounding-box height)",
               "edge_length_variance": "variance of the edge lengths",
               "centroid_y": "northernmost centroid (largest centroid y coordinate)",
               "width": "horizontal extent (bounding-box width)", "vertex_count": "number of vertices"}


def summary_line(poly):
    """The deterministic property summary of Table 3.  It states PRIMITIVE
    measurements.  Composite quantities the pipeline may ask about (the aspect
    ratio = width / height) are deliberately not pre-computed: an augmented or
    handle+summary step still has to combine two stated numbers.  That is what a
    real framework summary looks like, and it means those conditions measure
    read-and-use rather than read-and-copy."""
    q = properties(poly)
    return ("[Properties]\n"
            f"  area = {q['area']}\n"
            f"  perimeter = {q['perimeter']}\n"
            f"  width = {q['width']}\n"
            f"  height = {q['height']}\n"
            f"  edge_length_variance = {q['edge_length_variance']}\n"
            f"  vertex_count = {q['vertex_count']}\n"
            f"  centroid = ({q['centroid'][0]}, {q['centroid'][1]})\n"
            f"  bbox = ({q['bbox'][0]}, {q['bbox'][1]}, {q['bbox'][2]}, {q['bbox'][3]})\n"
            f"  convex = {q['convex']}\n"
            f"  is_simple = {q['is_simple']}")


# ===========================================================================
# The four state-passing conditions (Table 3) and the two propagation modes
# ===========================================================================

CONDITIONS = ["raw", "augmented", "handle", "handle_sum"]
MODES = ["cascading", "oracle"]
TEXT_CONDITIONS = {"raw", "augmented"}
HANDLE_CONDITIONS = {"handle", "handle_sum"}


def operand_labels(visible_handles, condition):
    if condition in HANDLE_CONDITIONS:
        return {h: h for h in visible_handles}
    return {h: f"Polygon {chr(65 + i)}" for i, h in enumerate(visible_handles)}


def render_state(visible, condition):
    labels = operand_labels([h for h, _ in visible], condition)
    parts = []
    for h, p in visible:
        lab = labels[h]
        if condition == "raw":
            parts.append(f"{lab}:\n{to_wkt(p)}")
        elif condition == "augmented":
            parts.append(f"{lab}:\n{to_wkt(p)}\n{summary_line(p)}")
        elif condition == "handle":
            parts.append(lab)
        elif condition == "handle_sum":
            parts.append(f"{lab}\n{summary_line(p)}")
        else:
            raise ValueError(condition)
    return "\n\n".join(parts)


def render_instruction(template, visible_handles, condition):
    labels = operand_labels(visible_handles, condition)
    return template.format(**{f"o{i}": labels[h] for i, h in enumerate(visible_handles)})


def object_matches(ref, expected_poly, expected_handle, condition):
    """handle conditions: the id must match.  text conditions: the WKT must
    parse and agree with the expected geometry to 1% (area and bounds)."""
    if ref is None:
        return False, "missing"
    s = str(ref).strip()
    if condition in HANDLE_CONDITIONS:
        return (s == expected_handle), ("ok" if s == expected_handle else "wrong_handle")
    m = re.search(r"POLYGON\s*\(\(.*?\)\)", s, re.S | re.I)
    if not m:
        return False, "not_wkt"
    try:
        got = shapely_wkt.loads(m.group(0))
    except Exception:
        return False, "unparseable_wkt"
    if got.is_empty or not got.is_valid:
        return False, "invalid_wkt"
    if abs(got.area - expected_poly.area) / max(abs(expected_poly.area), 1e-9) > 0.01:
        return False, "wrong_geometry"
    b1, b2 = got.bounds, expected_poly.bounds
    span = max(b2[2] - b2[0], b2[3] - b2[1], 1e-9)
    if max(abs(x - y) for x, y in zip(b1, b2)) / span > 0.01:
        return False, "wrong_geometry"
    return True, "ok"

Overwriting exp2_runtime.py


In [5]:
import json, math, random, re, threading, time, traceback
import numpy as np
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

from shapely import wkt as shapely_wkt
from exp2_runtime import *      # noqa

## [CELL 3] Prompt and parsing

In [6]:
_OP_LINES = {
    "convex_hull":    "  convex_hull(polygon)                  -> polygon: the convex hull",
    "buffer":         "  buffer(polygon, distance)             -> polygon: expanded outward by distance",
    "simplify":       "  simplify(polygon, tolerance)          -> polygon: vertices removed within tolerance",
    "bounding_box":   "  bounding_box(polygon)                 -> polygon: axis-aligned bounding box",
    "rotate":         "  rotate(polygon, angle)                -> polygon: rotated by angle degrees about its centroid",
    "translate":      "  translate(polygon, dx, dy)            -> polygon: shifted by (dx, dy)",
    "scale":          "  scale(polygon, factor)                -> polygon: scaled by factor about its centroid",
    "intersection":   "  intersection(polygon_a, polygon_b)    -> polygon: the overlapping region",
    "union":          "  union(polygon_a, polygon_b)           -> polygon: the combined region",
    "centroid":       "  centroid(polygon)                     -> point: the area centroid",
    "triangulate":    "  triangulate(polygon)                  -> triangles: Delaunay triangulation",
    "voronoi":        "  voronoi(polygon)                      -> regions: Voronoi diagram",
    "nearest_points": "  nearest_points(polygon_a, polygon_b)  -> point pair: closest points",
}
_DESCRIBE_LINE = "  describe(polygon)                     -> text: a computed property summary of the polygon"
_GET_OBJECT_LINE = "  get_object(polygon_id)                -> text: the WKT of that object (no properties)"
GET_OBJECT_TOOL = "get_object"

FORMAT_RULE = ("You must always answer with exactly one tool call, even if you are unsure; never ask a question "
               "or leave the action out.\n"
               "Respond with exactly two lines.\n"
               "Thought: one short sentence.\n"
               'Action: {"tool": "<name>", "objects": [...], "args": [...]}')

def tool_spec(is_final_step, describe, get_object):
    names = CHAINABLE_OPS + (TERMINAL_OPS if is_final_step else [])
    lines = ["Available operations:"] + [_OP_LINES[n] for n in names]
    body = "\n".join(lines)
    body += ("\n\nEvery operation consumes the step: the call you make is the answer for this step.")
    if is_final_step:
        body += ("\ncentroid, triangulate, voronoi and nearest_points do not return a polygon, "
                 "so they are available only on this final step.")
    reads = ([_GET_OBJECT_LINE] if get_object else []) + ([_DESCRIBE_LINE] if describe else [])
    if reads:
        body += ("\n\nREAD operation" + ("s" if len(reads) > 1 else "") + ", also available:\n" + "\n".join(reads) +
                 "\nA read does not change anything and does not consume the step: its result is "
                 "shown to you and the same task is asked again.")
    return body

def object_rule(condition):
    if condition in HANDLE_CONDITIONS:
        return 'Give each object argument as its id, for example "polygon_1".'
    return "Give each object argument as its complete WKT string, copied exactly as shown."

def system_prompt(condition, is_final_step, describe, get_object=False):
    return ("You are operating a geometry tool runtime. You receive the current state and a task, "
            "and you answer with one tool call.\n\n" + tool_spec(is_final_step, describe, get_object)
            + "\n\n" + object_rule(condition) + "\n\n" + FORMAT_RULE)

def user_turn(observation, task, condition):
    return f"Observation:\n{observation}\n\nTask:\n{task}\n\n{object_rule(condition)}"


def parse_react(text):
    """Return dict(tool, objects, args, thought, parse_path, parse_error)."""
    out = dict(tool=None, objects=[], args=[], thought=None, parse_path=None, parse_error=None)
    if not text:
        out["parse_error"] = "empty_response"; return out
    m = re.search(r"Thought:\s*(.*)", text)
    if m: out["thought"] = m.group(1).strip()[:300]
    body = re.sub(r"```(?:json)?", "", text)
    cand = None
    m = re.search(r"Action:\s*(\{.*)", body, re.S)
    if m:
        cand = m.group(1); out["parse_path"] = "action_line"
    else:
        m = re.search(r"(\{[^{}]*\"tool\"[^{}]*\})", body, re.S)
        if m: cand = m.group(1); out["parse_path"] = "bare_json"
    if cand is None:
        out["parse_error"] = "no_action_json"; return out
    obj = None
    # take the first balanced {...} object
    depth = 0; start = None
    for i, ch in enumerate(cand):
        if ch == "{":
            if depth == 0: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                try:
                    obj = json.loads(cand[start:i + 1]); break
                except Exception:
                    start = None
    if not isinstance(obj, dict):
        out["parse_error"] = "unparseable_json"; return out
    tool = str(obj.get("tool", "")).strip().lower()
    objs = obj.get("objects", obj.get("object", []))
    if isinstance(objs, str): objs = [objs]
    args = obj.get("args", [])
    if not isinstance(args, list): args = [args]
    out["objects"] = [str(o) for o in (objs or [])]
    out["args"] = args
    if tool in (DESCRIBE_TOOL, GET_OBJECT_TOOL):
        out["tool"] = tool
    elif tool in VALID_TOOLS:
        out["tool"] = tool
    else:
        out["parse_error"] = f"unknown_operation:{tool}"
    return out

VALID_TOOLS = set(OPS)

## [CELL 4] Scoring

`correct` = right tool, right arguments, right operands. Gated steps add
`decision_correct` (the branch or the chosen object) and `deferral` (a call
outside `allowed_tools`).

In [7]:
ARG_TOL_STATED  = 1e-6   # an argument printed in the instruction must be copied
ARG_TOL_DERIVED = 0.05   # an argument the model had to compute (Table 2, derived mechanism)
READ_TOOLS = {DESCRIBE_TOOL, GET_OBJECT_TOOL}

def score_step(step, p, store, condition):
    exp_tool, exp_args = step["correct_tool"], step["correct_args"]
    tool_ok = (p["tool"] == exp_tool)
    tol = ARG_TOL_DERIVED if step.get("mechanism") == "derived" else ARG_TOL_STATED
    args_ok, errs = True, []
    try:
        got_args = [float(a) for a in p["args"]]
    except (TypeError, ValueError):
        got_args, args_ok = [], False
    if tool_ok and exp_args and args_ok:
        if len(got_args) != len(exp_args):
            args_ok = False
        else:
            for g, w in zip(got_args, exp_args):
                e = abs(g - w) / abs(w) if abs(w) > 1e-9 else abs(g - w)
                errs.append(e)
                if e > tol: args_ok = False
    elif tool_ok and not exp_args:
        args_ok = (len(got_args) == 0)
    exp_ops = step["correct_operands"]
    if len(p["objects"]) != len(exp_ops):
        obj_ok, why = False, "wrong_object_count"
    else:
        obj_ok, why = True, "ok"
        for ref, h in zip(p["objects"], exp_ops):
            ok, w = object_matches(ref, store[h], h, condition)
            if not ok:
                obj_ok, why = False, w; break
    call_ok = bool(tool_ok and args_ok)
    allowed = step.get("allowed_tools") or [exp_tool]
    deferral = bool(p["tool"] is not None and p["tool"] not in READ_TOOLS and p["tool"] not in allowed)
    if step.get("mechanism") == "threshold":
        decision_ok = bool(tool_ok)
    elif step.get("mechanism") == "boolean":
        decision_ok = bool(tool_ok and args_ok)      # both branches use the same tool; the branch is in the args
    elif step.get("mechanism") == "selection":
        decision_ok = bool(len(p["objects"]) == 1 and object_matches(p["objects"][0], store[exp_ops[0]], exp_ops[0], condition)[0])
    elif step.get("mechanism") == "derived":
        decision_ok = bool(tool_ok and args_ok)
    else:
        decision_ok = None
    return dict(tool_ok=tool_ok, args_ok=args_ok, objects_ok=obj_ok, object_failure=why, arg_tolerance=tol,
                call_correct=call_ok, correct=bool(call_ok and obj_ok), decision_correct=decision_ok,
                deferral=deferral, parse_failure=bool(p["tool"] is None),
                max_arg_rel_error=max(errs) if errs else None)


def resolve_operand(ref, visible, condition):
    """Map the model's object reference to a stored handle (cascading mode)."""
    if condition in HANDLE_CONDITIONS:
        return ref if ref in dict(visible) else None
    for h, poly in visible:
        if object_matches(ref, poly, h, condition)[0]:
            return h
    return None


def try_execute(p, step, store, condition):
    if p["tool"] is None or p["tool"] in READ_TOOLS:
        return False, None, "no_valid_tool"
    visible = [(h, store[h]) for h in step["visible_handles"] if h in store]
    hs = [resolve_operand(r, visible, condition) for r in p["objects"]]
    if any(h is None for h in hs):
        return False, None, "unresolvable_operand"
    try:
        args = [float(a) for a in p["args"]]
        res = apply_op(p["tool"], [store[h] for h in hs], args)
    except Exception as e:
        return False, None, f"execution_error:{type(e).__name__}"
    if not is_terminal(p["tool"]) and not is_runtime_valid(res):
        return False, None, "invalid_result"
    return True, res, None

## [CELL 5] Model clients

`openai_compatible` covers OpenAI, DeepSeek, OpenRouter (Qwen, Llama) and
any other endpoint with the same API. `google` uses the `google-genai` SDK.
The two mock clients exercise every code path without an API key:
`mock_oracle` always answers correctly, `mock_random` answers badly on
purpose (deferrals, wrong branches, malformed JSON).

In [8]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type, retry_if_not_exception_type
import re as _re, threading as _threading

class FatalBackendError(RuntimeError):
    """Balance / quota / billing failure.  Retrying cannot help: the whole run must stop."""

STOP_EVENT = _threading.Event()   # set once a fatal error is seen; every worker checks it before each LLM call
STOP_REASON = []                  # the first fatal error message
_FATAL_RE = _re.compile(
    r"insufficient[ _]?(balance|credit|credits|quota|funds)"      # DeepSeek 402 body is exactly "Insufficient Balance"
    r"|payment required|billing"
    r"|exceeded your current quota|quota exceeded"
    r"|out of credits?|credit balance is too low|no remaining credit"
    r"|account (is )?(suspended|disabled|deactivated)"
    r"|arrearage|余额不足|欠费"
    r"|free-models-per-day|requests per day|daily limit|daily quota"   # a daily cap: retrying cannot help
    r"|add (more )?credits|requires more credits|upstream .*quota",
    _re.I)

def _is_fatal(e):
    """True only for money/quota failures.  Parse and structural errors never reach here."""
    if getattr(e, "status_code", None) == 402:                   # canonical "Payment Required"
        return True
    # a 429 can mean either "slow down" (retry) or "you are out of quota" (fatal) - the text decides
    return bool(_FATAL_RE.search(str(e)))

def _check_stop():
    if STOP_EVENT.is_set():
        raise FatalBackendError("run stopped: a fatal backend error (insufficient balance / quota) was seen by another worker")

def _guard(fn):
    """Run one provider call.  A balance/quota error sets STOP_EVENT and becomes FatalBackendError."""
    _check_stop()
    try:
        return fn()
    except FatalBackendError:
        raise
    except Exception as e:
        if _is_fatal(e):
            if not STOP_EVENT.is_set(): STOP_REASON.append(f"{type(e).__name__}: {e}")
            STOP_EVENT.set()
            raise FatalBackendError(f"{type(e).__name__}: {e}") from e
        raise

class OpenAICompatible:
    def __init__(self, model, base_url, api_key):
        from openai import OpenAI, APITimeoutError, RateLimitError, APIConnectionError, InternalServerError
        self.client = OpenAI(api_key=api_key, base_url=base_url, timeout=120)
        self.model = model
        self._retry_on = (APITimeoutError, RateLimitError, APIConnectionError, InternalServerError)
        self._RateLimitError = RateLimitError
    def __call__(self, messages, hint=None):
        @retry(stop=stop_after_attempt(4), wait=wait_exponential(multiplier=2, min=2, max=30),
               retry=retry_if_exception_type(self._retry_on), reraise=True)
        def go():
            kw = dict(model=self.model, messages=messages, temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
            if EXTRA_BODY: kw["extra_body"] = EXTRA_BODY
            r = _guard(lambda: self.client.chat.completions.create(**kw))
            u = r.usage; msg = r.choices[0].message
            det = getattr(u, "completion_tokens_details", None)
            rtok = getattr(det, "reasoning_tokens", None) if det is not None else None
            rc = getattr(msg, "reasoning_content", None) or getattr(msg, "reasoning", None)
            return msg.content or "", dict(input_tokens=getattr(u, "prompt_tokens", 0),
                                          output_tokens=getattr(u, "completion_tokens", 0),
                                          reasoning_tokens=rtok, hidden_reasoning_chars=len(rc) if rc else 0,
                                          finish_reason=r.choices[0].finish_reason)
        try:
            return go()
        except self._RateLimitError as e:
            # tenacity already retried 4x with backoff and the limit is still there.
            # A transient spike clears in seconds; this one did not, so it is a wall
            # (daily cap / exhausted credits) and the whole run should stop.
            if globals().get("STOP_ON_RATE_LIMIT", False):
                if not STOP_EVENT.is_set():
                    STOP_REASON.append(f"rate limit survived 4 retries: {e}")
                STOP_EVENT.set()
                raise FatalBackendError(f"persistent rate limit: {e}") from e
            raise

class GoogleClient:
    def __init__(self, model, api_key):
        import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], check=False)
        from google import genai
        self.genai = genai; self.client = genai.Client(api_key=api_key); self.model = model
    def __call__(self, messages, hint=None):
        from google.genai import types
        sys_txt = "\n".join(m["content"] for m in messages if m["role"] == "system")
        contents = [types.Content(role="user" if m["role"] == "user" else "model",
                                  parts=[types.Part(text=m["content"])]) for m in messages if m["role"] != "system"]
        @retry(stop=stop_after_attempt(4), wait=wait_exponential(multiplier=2, min=2, max=30),
               retry=retry_if_not_exception_type(FatalBackendError), reraise=True)
        def go():
            r = _guard(lambda: self.client.models.generate_content(
                model=self.model, contents=contents,
                config=types.GenerateContentConfig(system_instruction=sys_txt, temperature=TEMPERATURE,
                                                   max_output_tokens=MAX_TOKENS,
                                                   thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET))))
            um = getattr(r, "usage_metadata", None)
            fr = None
            try: fr = str(r.candidates[0].finish_reason)
            except Exception: pass
            return r.text or "", dict(input_tokens=getattr(um, "prompt_token_count", 0),
                                      output_tokens=getattr(um, "candidates_token_count", 0),
                                      reasoning_tokens=getattr(um, "thoughts_token_count", None), hidden_reasoning_chars=0,
                                      finish_reason=fr)
        return go()

class MockOracle:
    """Answers correctly.  `hint` carries the step, store and condition."""
    def __call__(self, messages, hint=None):
        step, store, cond = hint["step"], hint["store"], hint["condition"]
        objs = [h if cond in HANDLE_CONDITIONS else to_wkt(store[h]) for h in step["correct_operands"]]
        act = json.dumps({"tool": step["correct_tool"], "objects": objs, "args": step["correct_args"]})
        return f"Thought: I follow the instruction.\nAction: {act}", dict(input_tokens=0, output_tokens=0, finish_reason="stop")

class MockRandom:
    def __init__(self, seed=0): self.rng = random.Random(seed)
    def __call__(self, messages, hint=None):
        step, store, cond = hint["step"], hint["store"], hint["condition"]
        r = self.rng.random()
        objs = [h if cond in HANDLE_CONDITIONS else to_wkt(store[h]) for h in step["visible_handles"]]
        if r < 0.05:
            return "", dict(input_tokens=0, output_tokens=MAX_TOKENS, finish_reason="length")   # hidden reasoning hit the cap
        if r < 0.10:
            return "Thought: hmm\nAction: not json at all", dict(input_tokens=0, output_tokens=0, finish_reason="stop")
        if r < 0.25:
            act = {"tool": "bounding_box", "objects": objs[:1], "args": []}          # deferral
        elif r < 0.35 and cond in HANDLE_CONDITIONS:
            act = {"tool": "get_object", "objects": objs[:1], "args": []}             # fetch the WKT
        elif r < 0.30 and hint.get("describe"):
            act = {"tool": "describe", "objects": objs[:1], "args": []}
        elif step.get("mechanism") == "boolean":
            k = [h if cond in HANDLE_CONDITIONS else to_wkt(store[h]) for h in step["correct_operands"]]
            act = {"tool": "translate", "objects": k, "args": self.rng.choice([[20, 0], [0, 20]])}
        elif step.get("mechanism") == "threshold":
            t = self.rng.choice(["scale", "buffer"])
            args = [0.85] if t == "scale" else [float(re.search(r"buffer\([^,]+, ([\d.]+)\)", step["instruction"]).group(1))]
            k = [h if cond in HANDLE_CONDITIONS else to_wkt(store[h]) for h in step["correct_operands"]]
            act = {"tool": t, "objects": k, "args": args}
        elif step.get("mechanism") == "derived":
            noisy = [a * (1 + self.rng.uniform(-0.1, 0.1)) for a in step["correct_args"]]
            k = [h if cond in HANDLE_CONDITIONS else to_wkt(store[h]) for h in step["correct_operands"]]
            act = {"tool": step["correct_tool"], "objects": k, "args": noisy}
        elif step.get("mechanism") == "selection":
            act = {"tool": step["correct_tool"], "objects": [self.rng.choice(objs)], "args": step["correct_args"]}
        else:
            k = [h if cond in HANDLE_CONDITIONS else to_wkt(store[h]) for h in step["correct_operands"]]
            act = {"tool": step["correct_tool"], "objects": k, "args": step["correct_args"]}
        return f"Thought: guessing\nAction: {json.dumps(act)}", dict(input_tokens=0, output_tokens=0, finish_reason="stop")


def make_client():
    if PROVIDER == "mock_oracle": return MockOracle()
    if PROVIDER == "mock_random": return MockRandom()
    key = _get_key()
    if not key: raise RuntimeError(f"no API key in env/secret {API_KEY_ENV}")
    if PROVIDER == "openai_compatible": return OpenAICompatible(MODEL, BASE_URL, key)
    if PROVIDER == "google": return GoogleClient(MODEL, key)
    raise ValueError(PROVIDER)

## [CELL 6] One unit = one pipeline x one condition x one mode

In [9]:
def describe_budget(n_visible):
    return n_visible + 1

def run_unit(pl, condition, mode, call_llm, describe, get_object):
    store = {o["handle"]: shapely_wkt.loads(o["wkt"]) for o in pl["initial_objects"]}
    steps_out, transcript, io_log = [], [], []
    aborted = None
    n = len(pl["steps"])
    for si, step in enumerate(pl["steps"]):
        is_final = (si == n - 1)
        visible = [(h, store[h]) for h in step["visible_handles"] if h in store]
        obs = render_state(visible, condition)
        task = render_instruction(step["instruction"], step["visible_handles"], condition)
        go_here = bool(get_object and condition in HANDLE_CONDITIONS)
        budget = describe_budget(len(visible)) if describe else 0
        go_budget = len(visible) if go_here else 0
        d_used, d_log, g_used, g_log = 0, [], 0, []
        _check_stop()                      # another worker hit a fatal (balance/quota) error: stop this unit now
        msgs = [{"role": "system", "content": system_prompt(condition, is_final, describe, go_here)}] + transcript
        t0 = time.time(); tin = tout = 0; text = ""; parsed = None; rtok = 0; fin = None; hid = 0
        while True:
            msgs.append({"role": "user", "content": user_turn(obs, task, condition)})
            text, meta = call_llm(msgs, hint=dict(step=step, store=store, condition=condition, describe=describe))
            tin += meta.get("input_tokens") or 0; tout += meta.get("output_tokens") or 0
            rtok += meta.get("reasoning_tokens") or 0; hid += meta.get("hidden_reasoning_chars") or 0; fin = meta.get("finish_reason")
            msgs.append({"role": "assistant", "content": text or ""})
            io_log.append({"step": si, "messages": msgs[-2:], "response": text})
            parsed = parse_react(text)
            if parsed["tool"] == GET_OBJECT_TOOL:
                if go_here and g_used < go_budget:
                    g_used += 1
                    target = parsed["objects"][0] if parsed["objects"] and parsed["objects"][0] in dict(visible) else None
                    if target is None:
                        obs = "That read could not be performed: unknown object id.\nThe state is unchanged."
                        g_log.append({"n": g_used, "object": parsed["objects"], "ok": False})
                    else:
                        obs = f"{target}:\n{to_wkt(store[target])}\nThe state is unchanged."
                        g_log.append({"n": g_used, "object": target, "ok": True})
                    continue
                parsed = dict(parsed, tool=None, parse_error=("get_object_budget_exhausted" if go_here else "unknown_operation:get_object"))
            if describe and parsed["tool"] == DESCRIBE_TOOL:
                if d_used < budget:
                    d_used += 1
                    target = resolve_operand(parsed["objects"][0], visible, condition) if parsed["objects"] else None
                    if target is None:
                        obs = "That read could not be performed: unknown object.\nThe state is unchanged."
                        d_log.append({"n": d_used, "object": parsed["objects"], "ok": False})
                    else:
                        obs = f"{operand_labels([h for h, _ in visible], condition)[target]}\n{summary_line(store[target])}\nThe state is unchanged."
                        d_log.append({"n": d_used, "object": target, "ok": True})
                    continue
                parsed = dict(parsed, tool=None, parse_error="describe_budget_exhausted")
            break
        if parsed["tool"] is None and (text or "").strip() == "" and (fin in ("length", "max_tokens", "MAX_TOKENS") or tout >= MAX_TOKENS - 2):
            parsed = dict(parsed, parse_error="budget_exhausted")      # hidden reasoning hit max_tokens: not a parse failure
        sc = score_step(step, parsed, store, condition)
        rec = dict(index=si, kind=step["kind"], mechanism=step.get("mechanism"), gate_property=step.get("gate_property"),
                   decision_margin=step.get("decision_margin"), margin_band=step.get("margin_band"),
                   allowed_tools=step.get("allowed_tools"), is_noop=bool(step.get("is_noop")),
                   is_terminal=bool(step.get("is_terminal")), task=task, observation_chars=len(obs),
                   raw_response=text, thought=parsed["thought"], model_tool=parsed["tool"],
                   model_objects=parsed["objects"], model_args=parsed["args"], parse_path=parsed["parse_path"],
                   parse_error=parsed["parse_error"], expected_tool=step["correct_tool"],
                   expected_operands=step["correct_operands"], expected_args=step["correct_args"],
                   finish_reason=fin, reasoning_tokens=rtok, hidden_reasoning_chars=hid,
                   budget_exhausted=(parsed["parse_error"] == "budget_exhausted"),
                   describe_calls=d_used, describe_budget=budget, describe_log=d_log,
                   get_object_calls=g_used, get_object_budget=go_budget, get_object_log=g_log,
                   aux_calls=d_used + g_used, scored=True, input_tokens=tin, output_tokens=tout,
                   latency_ms=int((time.time() - t0) * 1000), **sc)
        transcript.append({"role": "user", "content": user_turn(obs, task, condition)})
        transcript.append({"role": "assistant", "content": text or ""})
        if mode == "oracle":
            res = apply_op(step["correct_tool"], [store[h] for h in step["correct_operands"]], step["correct_args"])
            if step["output_handle"]: store[step["output_handle"]] = res
            rec["execution_error"] = None; steps_out.append(rec)
        else:
            ok, res, why = try_execute(parsed, step, store, condition)
            rec["execution_error"] = why; steps_out.append(rec)
            if not ok:
                aborted = f"{'parse failure' if parsed['tool'] is None else 'malformed call'}: {why}"; break
            if step["output_handle"]: store[step["output_handle"]] = res
    final_ok = None
    ef = pl.get("expected_final")
    if ef and mode == "cascading" and aborted is None and ef["handle"] in store:
        try: final_ok = abs(store[ef["handle"]].area - ef["area"]) / max(abs(ef["area"]), 1e-9) <= 1e-3
        except Exception: final_ok = False
    gated = [s for s in steps_out if s["mechanism"] in ("threshold", "boolean", "derived", "selection")]
    return dict(
        pipeline_id=pl["pipeline_id"], category=pl["category"], gate_property=pl["gate_property"],
        gate_locality=pl.get("gate_locality"), margin_band=pl.get("margin_band"), depth=pl["depth"], tier=pl["tier"], shape_type=pl["shape_type"],
        condition=condition, mode=mode, n_steps=n, steps_scored=len(steps_out),
        steps_correct=sum(1 for s in steps_out if s["correct"]),
        decisions_scored=len(gated), decisions_correct=sum(1 for s in gated if s["decision_correct"]),
        deferrals=sum(1 for s in steps_out if s["deferral"]),
        aux_calls_total=sum(s["describe_calls"] + s["get_object_calls"] for s in steps_out),
        get_object_calls_total=sum(s["get_object_calls"] for s in steps_out),
        calls_correct=all(s["call_correct"] for s in steps_out) and len(steps_out) == n,
        end_to_end_correct=all(s["correct"] for s in steps_out) and len(steps_out) == n,
        final_state_matches_reference=final_ok,
        first_incorrect_step=next((s["index"] for s in steps_out if not s["correct"]), None),
        aborted=aborted, ends_terminal=bool(pl["steps"][-1].get("is_terminal")), steps=steps_out), io_log

## [CELL 7] Run: resumable, parallel, append-only

In [10]:
def load_pipelines(path, smoke):
    d = json.load(open(path))
    pls = d["pipelines"]
    if d.get("read_tools_in_primary_conditions", False):
        raise RuntimeError("this pipeline file was built for the v1 harness (read tools in primary conditions); rebuild with Phase 1 v2")
    if smoke:
        by = defaultdict(list)
        for p in pls: by[(p["category"], p["depth"], p.get("margin_band"))].append(p)
        groups = [v[0] for k, v in sorted(by.items(), key=str)]
        cats = defaultdict(list)
        for p in groups: cats[p["category"]].append(p)
        pls, i = [], 0
        while len(pls) < min(smoke, len(groups)):
            for c in sorted(cats):
                if i < len(cats[c]) and len(pls) < smoke: pls.append(cats[c][i])
            i += 1
    return d, pls

def main():
    tag = RUN_TAG or re.sub(r"[^A-Za-z0-9_.-]", "_", MODEL)
    out = Path("results") / tag; out.mkdir(parents=True, exist_ok=True)
    jsonl, jpath = out / "results.jsonl", out / "results.json"
    errs, iolog = out / "errors.md", out / "llm_io.jsonl"
    meta, pls = load_pipelines(DATASET, SMOKE)
    describe = bool(EXP4_DESCRIBE)
    protocol = "react_describe_v1" if describe else "react_v2"
    if describe and set(RUN_CONDITIONS) - {"raw", "handle"}:
        print("WARNING: Experiment 4 is defined for raw and handle only")
    done = set()
    if jsonl.exists() and not FORCE:
        for line in open(jsonl):
            try:
                r = json.loads(line)
                if (r.get("aborted") or "").startswith("backend error"):
                    continue                          # a backend error (rate limit, balance, timeout) is re-run, not kept
                done.add((r["pipeline_id"], r["condition"], r["mode"]))
            except Exception:
                pass
    units = [(p, c, m) for p in pls for c in RUN_CONDITIONS for m in RUN_MODES if (p["pipeline_id"], c, m) not in done]
    print(f"{len(pls)} pipelines x {len(RUN_CONDITIONS)} conditions x {len(RUN_MODES)} modes = {len(pls)*len(RUN_CONDITIONS)*len(RUN_MODES)} units; "
          f"{len(done)} already done; running {len(units)}")
    print(f"model={MODEL}  workers={WORKERS}  max_tokens={MAX_TOKENS}  out=results/{tag}/")
    client = make_client()
    lock = threading.Lock(); counters = Counter(); t_start = time.time()
    stopped = None

    def work(u):
        p, c, m = u
        if STOP_EVENT.is_set():
            return None                                   # run already stopping: leave this unit for the next run
        try:
            rec, io = run_unit(p, c, m, client, describe, GET_OBJECT_IN_HANDLE_CONDITIONS)
            return rec, io, None, False
        except Exception as e:
            fatal = isinstance(e, FatalBackendError)
            if fatal: STOP_EVENT.set()
            return dict(pipeline_id=p["pipeline_id"], category=p["category"], gate_property=p["gate_property"],
                        gate_locality=p.get("gate_locality"), margin_band=p.get("margin_band"), depth=p["depth"], tier=p["tier"], shape_type=p["shape_type"],
                        condition=c, mode=m, n_steps=len(p["steps"]), steps_scored=0, steps_correct=0,
                        decisions_scored=0, decisions_correct=0, deferrals=0, aux_calls_total=0, get_object_calls_total=0,
                        calls_correct=False, end_to_end_correct=False, final_state_matches_reference=None,
                        first_incorrect_step=None, aborted=f"backend error: {type(e).__name__}: {e}"[:300],
                        ends_terminal=bool(p["steps"][-1].get("is_terminal")), steps=[]), [], traceback.format_exc(), fatal

    with ThreadPoolExecutor(max_workers=WORKERS) as ex, open(jsonl, "a") as fj, open(iolog, "a") as fio, open(errs, "a") as fe:
        futs = [ex.submit(work, u) for u in units]
        fatal_msg = None
        for i, f in enumerate(as_completed(futs), 1):
            if f.cancelled() or f.result() is None:
                continue                              # queued unit dropped after a fatal error; it is re-run next time
            rec, io, tb, fatal = f.result()
            rec.update(model=MODEL, provider=PROVIDER, temperature=TEMPERATURE, thinking=THINKING, protocol=protocol,
                       dataset=Path(DATASET).name, timestamp=datetime.now(timezone.utc).isoformat())
            with lock:
                fj.write(json.dumps(rec) + "\n"); fj.flush()
                if LOG_LLM_IO and io:
                    fio.write(json.dumps({"pipeline_id": rec["pipeline_id"], "condition": rec["condition"], "mode": rec["mode"], "io": io}) + "\n")
                if tb:
                    fe.write(f"## {rec['pipeline_id']} {rec['condition']} {rec['mode']}\n```\n{tb}\n```\n"); counters["backend_errors"] += 1
                counters["units"] += 1; counters["steps"] += rec["steps_scored"]; counters["correct"] += rec["steps_correct"]
                counters["deferrals"] += rec["deferrals"]; counters["aborted"] += bool(rec["aborted"])
                if i % 25 == 0 or i == len(units):
                    el = time.time() - t_start
                    try:
                        _live = len(ex._threads)          # threads the pool has actually spawned
                    except Exception:
                        _live = WORKERS
                    print(f"[{i}/{len(units)}] workers {_live}/{WORKERS} "
                          f"step-acc {counters['correct']/max(counters['steps'],1):.3f} "
                          f"deferrals {counters['deferrals']} aborted {counters['aborted']} "
                          f"errors {counters['backend_errors']} {el/60:.1f} min")
            if fatal and fatal_msg is None:
                fatal_msg = STOP_REASON[0] if STOP_REASON else rec["aborted"]
                n_cancel = sum(1 for g in futs if g.cancel())
                print(f"\n!! FATAL backend error: {fatal_msg}\n"
                      f"   Stopping the run: {n_cancel} queued units cancelled, waiting for in-flight workers to exit...")
    export(out, meta, protocol)
    if fatal_msg:
        raise FatalBackendError(f"RUN STOPPED after {counters['units']} units: {fatal_msg}\n"
                                "Top up the account balance / quota, then re-run this cell: finished units are kept, "
                                "backend-error units are resumed automatically.")

def export(out, meta, protocol):
    latest = {}
    n_raw = 0
    for line in open(out / "results.jsonl"):
        try:
            r = json.loads(line); n_raw += 1; k = (r["pipeline_id"], r["condition"], r["mode"])
            if k in latest and (r.get("aborted") or "").startswith("backend error") and not (latest[k].get("aborted") or "").startswith("backend error"):
                continue                              # never let a backend error overwrite a real record
            latest[k] = r
        except Exception:
            pass
    recs = list(latest.values())
    json.dump(recs, open(out / "results.json", "w"))
    manifest = dict(model=MODEL, provider=PROVIDER, protocol=protocol, dataset=DATASET, dataset_version=meta.get("version"),
                    conditions=RUN_CONDITIONS, modes=RUN_MODES, exp4_describe=bool(EXP4_DESCRIBE),
                    thinking=THINKING, extra_body=EXTRA_BODY, thinking_budget=THINKING_BUDGET,
                    read_tools_in_primary_conditions=False, get_object_in_handle_conditions=bool(GET_OBJECT_IN_HANDLE_CONDITIONS),
                    n_pipelines=len(meta["pipelines"]),
                    units_recorded=len(recs), temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                    finished=datetime.now(timezone.utc).isoformat())
    json.dump(manifest, open(out / "manifest.json", "w"), indent=1)
    write_summary(out, recs, n_raw)
    print("wrote", out / "results.json", out / "manifest.json", out / "summary_report.txt")

def write_summary(out, recs, n_raw):
    L = [f"{MODEL} - Experiment 2 v2 summary", f"Generated (UTC): {datetime.now(timezone.utc).isoformat()}", "",
         f"Raw JSONL rows: {n_raw}; latest units retained: {len(recs)}", ""]
    L.append("condition / mode | units | clean | e2e correct | aborted | backend errors | deferral steps")
    L.append("-" * 96)
    for c in RUN_CONDITIONS:
        for m in RUN_MODES:
            u = [r for r in recs if r["condition"] == c and r["mode"] == m]
            be = sum(1 for r in u if (r["aborted"] or "").startswith("backend"))
            ab = sum(1 for r in u if r["aborted"]) - be
            clean = [r for r in u if not r["aborted"]]
            L.append(f"{c:10s} / {m:9s} | {len(u):5d} | {len(clean):5d} | {sum(r['end_to_end_correct'] for r in clean):5d} | {ab:5d} | {be:5d} | {sum(r['deferrals'] for r in u):5d}")
    be = sum(1 for r in recs if (r["aborted"] or "").startswith("backend error"))
    if be:
        L += ["", f"!! {be} units ended in a backend error (balance, rate limit, timeout). Re-run this cell: they are resumed automatically."]
    steps_all = [s for r in recs for s in r["steps"]]
    if steps_all:
        vis = np.mean([len(s["raw_response"]) / 4 for s in steps_all]); otok = np.mean([s["output_tokens"] for s in steps_all])
        bx = np.mean([s.get("budget_exhausted", False) for s in steps_all])
        L += ["", f"Reasoning check: mean output tokens {otok:.0f} vs ~{vis:.0f} visible tokens; budget-exhausted steps {100*bx:.1f}%."]
        if otok > 3 * max(vis, 20) or bx > 0.02:
            L += ["!! Hidden reasoning appears to be ON (or max_tokens too small). The primary run must be direct-answer: set EXTRA_BODY / THINKING_BUDGET for this provider."]
    L += ["", "ORACLE per-step accuracy (category x condition) and deferral rate  [HEADLINE = global gates; local = control]", "-" * 96]
    for cat, loc in (("pass_through", None), ("property_conditioned", "global"), ("multi_object", "global"),
                     ("property_conditioned", "local"), ("multi_object", "local")):
        row = []
        for c in RUN_CONDITIONS:
            ss = [s for r in recs if r["mode"] == "oracle" and r["condition"] == c and r["category"] == cat
                  and (loc is None or r.get("gate_locality") == loc) for s in r["steps"]]
            if not ss:
                row.append(f"{c}=n/a"); continue
            acc = sum(s["correct"] for s in ss) / len(ss); dfr = sum(s["deferral"] for s in ss) / len(ss)
            fetch = sum(s["get_object_calls"] > 0 for s in ss) / len(ss)
            row.append(f"{c}={acc:.3f} (def {dfr:.2f}, fetch {fetch:.2f})")
        L.append(f"{cat:22s}{(' [' + loc + ']') if loc else '':9s} " + "  ".join(row))
    FLOOR = {"threshold": 0.50, "boolean": 0.50, "selection": 0.25, "derived": 0.00}
    L += ["", "ORACLE accuracy by MECHANISM x category, with chance floor  (the primary table)", "-" * 96]
    for mech in ("threshold", "boolean", "derived", "selection"):
        for cat in ("property_conditioned", "multi_object"):
            row = []
            for c in RUN_CONDITIONS:
                ss = [s for r in recs if r["mode"] == "oracle" and r["condition"] == c and r["category"] == cat
                      and r.get("gate_locality") == "global" for s in r["steps"] if s["mechanism"] == mech]
                row.append(f"{c}={sum(s['correct'] for s in ss)/len(ss):.3f}" if ss else f"{c}=n/a")
            if any(not v.endswith("n/a") for v in row):
                L.append(f"{mech:10s} {cat:22s} floor={FLOOR[mech]:.2f}  " + "  ".join(row))
    L += ["", "ORACLE gated decision accuracy by gate x band", "-" * 96]
    for cat in ("property_conditioned", "multi_object"):
      for gate in sorted({r["gate_property"] for r in recs if r["category"] == cat}):
        for band in sorted({r.get("margin_band") for r in recs if r["category"] == cat and r["gate_property"] == gate}, key=str):
            row = []
            for c in RUN_CONDITIONS:
                ss = [s for r in recs if r["mode"] == "oracle" and r["condition"] == c and r["category"] == cat and r.get("margin_band") == band
                      and r["gate_property"] == gate for s in r["steps"] if s["decision_correct"] is not None]
                row.append(f"{c}={sum(s['decision_correct'] for s in ss)/len(ss):.3f}" if ss else f"{c}=n/a")
            L.append(f"{cat:22s} {gate:13s} {str(band):8s} " + "  ".join(row))
    (out / "summary_report.txt").write_text("\n".join(L))
    print("\n".join(L))

main()

924 pipelines x 4 conditions x 2 modes = 7392 units; 0 already done; running 7392
model=gpt-4.1-mini  workers=6  max_tokens=4096  out=results/gpt-4.1-mini/
[25/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 0.2 min
[50/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 0.2 min
[75/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 0.3 min
[100/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 0.4 min
[125/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 0.5 min
[150/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 0.6 min
[175/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 0.7 min
[200/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 0.9 min
[225/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 1.1 min
[250/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 1.3 min
[275/7392] workers 6/6 step-acc 1.000 deferrals 0 aborted 0 errors 0 1.4 min


FatalBackendError: RUN STOPPED after 1034 units: RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
Top up the account balance / quota, then re-run this cell: finished units are kept, backend-error units are resumed automatically.